In [1]:
# ===============================================
# GOOGLE COLAB WORKING VERSION (AUTO PATH FIX)
# ===============================================

!pip install -q kagglehub tensorflow

import os
import kagglehub
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# ===============================================
# 1. Download Dataset
# ===============================================

path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
print("Dataset downloaded at:", path)

# Find dataset root automatically
for root, dirs, files in os.walk(path):
    if "train" in dirs and "valid" in dirs:
        dataset_folder = root
        break

print("Dataset folder found at:", dataset_folder)

train_dir = os.path.join(dataset_folder, "train")
valid_dir = os.path.join(dataset_folder, "valid")

print("Train path:", train_dir)
print("Valid path:", valid_dir)

# ===============================================
# 2. Parameters
# ===============================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 38
EPOCHS = 5

# ===============================================
# 3. Data Generators
# ===============================================

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1
)

valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

valid_gen = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# ===============================================
# 4. Load MobileNetV2
# ===============================================

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

for layer in base_model.layers:
    layer.trainable = False

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dropout(0.35)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ===============================================
# 5. Callbacks
# ===============================================

checkpoint = ModelCheckpoint("mobilenetv2_best.keras",
                             monitor='val_accuracy',
                             save_best_only=True)

early_stop = EarlyStopping(monitor='val_loss',
                           patience=6,
                           restore_best_weights=True)

reduce_lr = ReduceLROnPlateau(monitor='val_loss',
                              factor=0.5,
                              patience=3)

# ===============================================
# 6. Train
# ===============================================

history = model.fit(
    train_gen,
    validation_data=valid_gen,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

# ===============================================
# 7. Evaluate
# ===============================================

loss, acc = model.evaluate(valid_gen)
print("Validation Accuracy:", acc)

# ===============================================
# 8. Save Model
# ===============================================

model.save("mobilenetv2_final.keras")
print("Training complete. Model saved.")

Using Colab cache for faster access to the 'new-plant-diseases-dataset' dataset.
Dataset downloaded at: /kaggle/input/new-plant-diseases-dataset
Dataset folder found at: /kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)
Train path: /kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train
Valid path: /kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid
Found 70295 images belonging to 38 classes.
Found 17572 images belonging to 38 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 1114s 498ms/step - accuracy: 0.4291 - loss: 2.1223 - val_accuracy: 0.8780 - val_loss: 0.4270 - learning_rate: 1.0000e-04
Epoch 2/5
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 840s 382ms/step - accuracy: 0.8043 - loss: 0.6271 - val_accuracy: 0.9030 - val_loss: 0.3129 - learning_rate: 1.0000e-04
Epoch 3/5
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 828s 377ms/step - accuracy: 0.8498 - loss: 0.4718 - val_accuracy: 0.9206 - val_loss: 0.2541 - learning_rate: 1.0000e-04
Epoch 4/5
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 834s 380ms/step - accuracy: 0.8722 - loss: 0.3993 - val_accuracy: 0.9199 - val_loss: 0.2480 - learning_rate: 1.0000e-04
Epoch 5/5
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 836s 380ms/step - accuracy: 0.8825 - loss: 0.3629 - val_accuracy: 0.9302 - val_loss: 0.2188 - learning_rate: 1.0000e-04
550/550 ━━━━━━━━━━━━━━━━━━━━ 34s 61ms/step - accuracy: 0.9317 - loss: 0.2169
Validation Accuracy: 0.9301729798316956
Training complete. Model saved.
